# ⚡ ChargeBot — ChargeGrid Intelligence
## GoodWe × FIAP · EV Challenge 2026 · Sprint 2
### Motor: Google Gemini (gratuito via AI Studio)

---
### ⚠️ Antes de rodar:
1. Acesse **aistudio.google.com/app/apikey** e crie uma chave gratuita
2. No painel esquerdo do Colab, clique no ícone 🔑 **(Secrets)**
3. Adicione: Nome = `GOOGLE_API_KEY` · Valor = sua chave
4. Ative o toggle ao lado do secret
5. Execute as células em ordem com **Ctrl+F9**

In [ ]:
# ── CÉLULA 1: Instalação ─────────────────────────────────────────────────
!pip install google-generativeai -q
print('✅ google-generativeai instalado')

In [ ]:
# ── CÉLULA 2: API Key via Colab Secrets (seguro, sem expor a chave) ──────
import os
from google.colab import userdata

os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
print('✅ API Key configurada com segurança')

In [ ]:
# ── CÉLULA 3: Imports e dados simulados da rede ───────────────────────────
import json
import random
from datetime import datetime, timedelta
import google.generativeai as genai

def get_network_status():
    pool = ['online']*5 + ['in_use']*3 + ['fault']*2 + ['offline']*2
    chargers = []
    for i in range(1, 13):
        s = random.choice(pool)
        c = {'id': f'CG-{i:02d}', 'status': s,
             'power_kw': round(random.uniform(7, 22), 1) if s == 'in_use' else 0,
             'sessions_today': random.randint(0, 12) if s != 'offline' else 0}
        if s == 'fault':
            c['error_code'] = random.choice(['E-04','E-07','E-12'])
            c['fault_since'] = (datetime.now()-timedelta(minutes=random.randint(10,120))).strftime('%H:%M')
        chargers.append(c)
    faults  = [c for c in chargers if c['status']=='fault']
    offline = [c for c in chargers if c['status']=='offline']
    return {'total':12,
            'online': sum(1 for c in chargers if c['status'] in ['online','in_use']),
            'fault':len(faults), 'offline':len(offline),
            'in_use': sum(1 for c in chargers if c['status']=='in_use'),
            'total_kw': round(sum(c['power_kw'] for c in chargers),1),
            'faults':faults, 'offline_units':offline}

def get_financials():
    rt = round(random.uniform(600,1100),2)
    return {'receita_hoje':rt, 'receita_ontem':round(random.uniform(700,1050),2),
            'media_diaria':round(random.uniform(780,870),2),
            'sessoes_hoje':random.randint(22,48),
            'duracao_media_min':random.randint(28,55),
            'projecao_mensal':round(rt*30,2), 'meta_mensal':22000.00}

def get_alerts():
    pool = [
        {'id':'ALT-001','hora':(datetime.now()-timedelta(minutes=23)).strftime('%H:%M'),
         'tipo':'power_spike','severidade':'medio','carregador':'CG-05',
         'mensagem':'Consumo 24kW — limite: 11kW','acao':'Verificar cabo e config CG-05'},
        {'id':'ALT-002','hora':(datetime.now()-timedelta(minutes=67)).strftime('%H:%M'),
         'tipo':'communication_loss','severidade':'alto','carregador':'CG-08',
         'mensagem':'Perda OCPP 67 min','acao':'Verificar rede CG-08'},
    ]
    return pool[:random.choice([0,0,1,2])]

def build_context():
    return json.dumps({
        'horario': datetime.now().strftime('%d/%m/%Y %H:%M'),
        'rede': get_network_status(),
        'financeiro': get_financials(),
        'alertas_ativos': get_alerts()
    }, ensure_ascii=False, indent=2)

print('✅ Dados simulados carregados')

In [ ]:
# ── CÉLULA 4: System Prompt + Few-shot + Configuração do Gemini ───────────
SYSTEM_PROMPT = """Você é o ChargeBot, assistente operacional da plataforma ChargeGrid Intelligence da GoodWe.

## Identidade
Especialista em redes de eletropostos (EVSE). Atende operadores comerciais. Tom profissional, direto e orientado a dados. Responde SEMPRE em Português do Brasil.

## Regras
SEMPRE: use dados do [CONTEXTO] injetado · cite IDs dos carregadores · compare dados financeiros com referências · ofereça próximo passo acionável · emojis moderados (✅⚠️❌📊💡).
NUNCA: invente dados ausentes · use tom alarmista · finalize conversa sobre falha sem próximo passo.

## Códigos de Erro GoodWe
E-04: RFID | E-07: OCPP/Comunicação | E-12: Sobrecarga | E-15: Temperatura | E-21: Relé

## Dashboard
Tarifas: Configurações→Tarifas→Nova Regra | Potência: Equipamentos→[ID]→Configurações | Relatórios: Relatórios→Exportar

=== EXEMPLOS ===
[Status] Usuário: Quantos carregadores online?
ChargeBot: **10/12** operando. ⚠️ CG-03 em falha (E-07). 🔧 CG-09 offline. Quer abrir chamado?

[Receita] Usuário: Receita hoje?
ChargeBot: R$ 847,50 em 34 sessões. vs. ontem: -8,2% · vs. média: +4,6%. Quer detalhes por carregador?

[Alerta] Usuário: Teve algum alerta?
ChargeBot: ⚠️ CG-05 às 16h47 — consumo 24kW (limite: 11kW). Verifique cabo e config de potência.

[Erro] Usuário: Erro E-04 no CG-03?
ChargeBot: E-04 = falha RFID. Causas: cartão danificado, leitor sujo, credencial expirada. Workaround: QR Code ou app.
=== FIM ==="""

genai.configure(api_key=os.environ.get('GOOGLE_API_KEY'))
model = genai.GenerativeModel(
    model_name='gemini-2.0-flash',
    system_instruction=SYSTEM_PROMPT,
    generation_config=genai.GenerationConfig(temperature=0.3, max_output_tokens=800)
)

# Histórico no formato do Gemini: role='user'|'model'
chat_history = []

def chargebot_chat(user_message):
    enriched = f"{user_message}\n\n[CONTEXTO]\n{build_context()}"
    session  = model.start_chat(history=chat_history)
    response = session.send_message(enriched)
    reply    = response.text.strip()
    # Salva no histórico apenas a mensagem original (sem bloco de contexto)
    chat_history.append({'role':'user',  'parts':[user_message]})
    chat_history.append({'role':'model', 'parts':[reply]})
    # Janela máxima: últimas 10 trocas (20 mensagens)
    if len(chat_history) > 20:
        del chat_history[:2]
    return reply

print('✅ Gemini configurado e pronto!')

In [ ]:
# ── CÉLULA 5: Interface visual interativa ────────────────────────────────
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets

output_area = widgets.Output()
text_input  = widgets.Text(
    placeholder='Digite sua pergunta sobre a rede de eletropostos...',
    layout=widgets.Layout(width='75%')
)
send_btn  = widgets.Button(description='Enviar ⚡', button_style='primary',
                           layout=widgets.Layout(width='12%'))
reset_btn = widgets.Button(description='Limpar 🗑️', button_style='warning',
                           layout=widgets.Layout(width='12%'))

log = []  # [(role, texto)]

def render():
    with output_area:
        clear_output(wait=True)
        html = ('<div style="font-family:monospace;background:#0d1117;padding:16px;'
                'border-radius:10px;max-height:480px;overflow-y:auto;border:1px solid #30363d">')
        html += ('<p style="color:#58a6ff;font-weight:bold;margin:0 0 8px">'
                 '⚡ ChargeBot · GoodWe ChargeGrid Intelligence · Gemini</p>'
                 '<hr style="border-color:#30363d;margin:0 0 12px">')
        if not log:
            html += '<p style="color:#8b949e">Nenhuma mensagem ainda. Faça sua pergunta abaixo!</p>'
        for role, msg in log:
            if role == 'user':
                html += (f'<p style="margin:6px 0"><b style="color:#58a6ff">Você:</b> '
                         f'<span style="color:#c9d1d9">{msg}</span></p>')
            else:
                safe = msg.replace('\n','<br>').replace('**','<b>',1).replace('**','</b>',1)
                html += (f'<div style="background:#161b22;border-left:3px solid #3fb950;'
                         f'padding:8px 12px;margin:6px 0;border-radius:4px">'
                         f'<b style="color:#3fb950">ChargeBot:</b><br>'
                         f'<span style="color:#c9d1d9">{safe}</span></div>')
        html += '</div>'
        display(HTML(html))

def on_send(b):
    msg = text_input.value.strip()
    if not msg:
        return
    text_input.value = ''
    log.append(('user', msg))
    render()
    with output_area:
        display(HTML('<p style="color:#8b949e;font-style:italic">⚡ ChargeBot digitando...</p>'))
    try:
        reply = chargebot_chat(msg)
    except Exception as e:
        reply = f'❌ Erro: {e}'
    log.append(('bot', reply))
    render()

def on_reset(b):
    log.clear()
    chat_history.clear()
    render()

send_btn.on_click(on_send)
reset_btn.on_click(on_reset)
text_input.on_submit(on_send)

display(widgets.VBox([
    widgets.HTML('<h3 style="color:#3fb950">⚡ ChargeBot — GoodWe EV Challenge 2026</h3>'),
    output_area,
    widgets.HBox([text_input, send_btn, reset_btn])
]))
render()

---
## 🧪 Execução Automática dos Casos de Teste (Sprint 1)

Rode a célula abaixo para testar os 6 casos automaticamente e ver os resultados.

In [ ]:
# ── CÉLULA 6: Casos de teste automáticos ─────────────────────────────────
test_cases = [
    ('CT-1 Status',       'Quantos carregadores estão online agora e tem algum com problema?'),
    ('CT-2 Receita',      'Qual foi a receita da rede hoje? Está dentro do esperado?'),
    ('CT-3 Alerta',       'O sistema mandou um alerta agora pouco, o que aconteceu?'),
    ('CT-4 Projeção',     'Me dá um resumo do desempenho e a projeção de receita do mês.'),
    ('CT-5 Configuração', 'Como configuro uma tarifa diferente para o horário de pico entre 18h e 20h?'),
    ('CT-6 Erro (bônus)', 'Apareceu o erro E-04 no carregador 3. O que é isso e o que faço?'),
]

chat_history.clear()
print('='*70)
print('EXECUÇÃO DOS CASOS DE TESTE — ChargeBot GoodWe · Sprint 2 · Gemini')
print('='*70)

for label, question in test_cases:
    print(f'\n[{label}]')
    print(f'Pergunta: {question}')
    print('-'*50)
    try:
        resp = chargebot_chat(question)
        print(f'Resposta ChargeBot:\n{resp}')
    except Exception as e:
        print(f'ERRO: {e}')
    print('='*70)